<a href="https://colab.research.google.com/github/DaviAlves06/Soccer-Rules-AI/blob/main/Agente_de_Regras_de_Futebol.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# PROJETO PRÁTICO - Agente de Regras de Futebol (RAG)

Conceitos abordados:
* Definição do problema
* Seleção da base de conhecimento
* Preparação dos documentos
* Embeddings e banco vetorial
* Recuperação de contexto (Retriever)
* Integração com LLM
* Testes e validação das respostas




## Dependências

```bash
!pip install -qU \
  "langchain>=0.2.0,<0.4.0" \
  langchain-community \
  langchain-google-genai \
  chromadb \
  pypdf \
  -qU sentence-transformers
```

In [ ]:
!pip install -qU \
  "langchain>=0.2.0,<0.4.0" \
  langchain-community \
  langchain-google-genai \
  chromadb \
  pypdf \
  -qU sentence-transformers

In [ ]:
# Importa o módulo para manipulação de variáveis de ambiente
import os

# Loader responsável por ler arquivos PDF
from langchain_community.document_loaders import PyPDFLoader

# Responsável por dividir textos grandes em chunks menores
from langchain.text_splitter import RecursiveCharacterTextSplitter

# Embeddings com Gemini (Google Generative AI)
from langchain_google_genai import GoogleGenerativeAIEmbeddings

# Banco vetorial para armazenamento e busca semântica
from langchain_community.vectorstores import Chroma

# Modelo de linguagem conversacional (Gemini)
from langchain_google_genai import ChatGoogleGenerativeAI

# Cadeia pronta de Perguntas e Respostas com RAG
from langchain.chains import RetrievalQA

In [ ]:
import os
import getpass

# Defina sua chave da Google AI (Gemini) como variável de ambiente
# Gere a chave em: https://ai.google.dev/gemini-api/docs/api-key
if not os.getenv("GOOGLE_API_KEY"):
    os.environ["GOOGLE_API_KEY"] = getpass.getpass("Informe sua GOOGLE_API_KEY: ")

##1️⃣ Definição do Problema

LLMs possuem conhecimento estático e podem alucinar. O objetivo aqui é garantir respostas confiáveis, conectando o modelo a documentos oficiais sobre regras do futebol.

## 2️⃣ Seleção e Organização das Bulas


In [ ]:
# Lista com os caminhos dos arquivos PDF
caminhos_regras = [
    "regras_futebol.pdf",
]

# Lista que armazenará todos os documentos carregados
documentos = []

# Percorre cada arquivo
for caminho in caminhos_regras:

    # Cria o loader do PDF
    loader = PyPDFLoader(caminho)

    # Carrega o conteúdo do PDF
    docs = loader.load()

    # Adiciona os documentos à lista principal
    documentos.extend(docs)

# Exibe a quantidade total de páginas carregadas
len(documentos)


## 3️⃣ Extração, Limpeza e Chunking



In [ ]:

# Cria o objeto responsável pelo chunking
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=600,      # Tamanho máximo de cada chunk
    chunk_overlap=150   # Sobreposição entre chunks
)

# Divide os documentos em chunks
chunks = text_splitter.split_documents(documentos)

# Quantidade total de chunks gerados
len(chunks)



## 4️⃣ Geração de Embeddings e Banco Vetorial


In [ ]:
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

# Embeddings locais (HuggingFace) – estáveis, gratuitos e sem depender de API externa
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

# Cria o banco vetorial com os chunks
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory="./chroma_regras"
)

##5️⃣ Recuperação de Contexto (Retriever)

In [ ]:

# Cria o retriever para busca semântica
retriever = vectorstore.as_retriever(
    search_kwargs={"k": 100}  # Número de chunks retornados
)



## 6️⃣ Integração com Pipeline RAG


In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0.1,   # respostas mais factuais
    transport="rest",  # evita gRPC e o erro 'Illegal metadata'
    timeout=60,        # tempo máx. de cada chamada (segundos)
    max_retries=1,     # no máx. 1 retry rápido; evita ficar 10+ minutos
)

# Cria a cadeia RAG
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=retriever,
    return_source_documents=True
)

## 7️⃣ Testes e Validação das Respostas


In [ ]:
# Pergunta de teste
pergunta = "O que acontece caso um jogador coloque a mão na bola?"

# Executa a pergunta no agente RAG (forma atual)
resposta = qa_chain.invoke(pergunta)

print("Pergunta:")
print(pergunta)

print("\nResposta do Agente:")
print(resposta["result"])

print("\nTrechos utilizados como contexto:\n")

# Percorre os documentos recuperados
for i, doc in enumerate(resposta["source_documents"], start=1):
    print(f"--- Trecho {i} ---")

    # Conteúdo recuperado
    print("\nConteúdo do chunk:")
    print(doc.page_content)
    print("\n")
